# Notebook 5: Feature Engineering
**MLOps Training 2026/2027 · Task 2: From Tables to Notebooks**

### Objective:
1. Load the three dataset splits (`train.parquet`, `val.parquet`, `test.parquet`).
2. Engineer new predictive features based on EDA findings (Notebook 4).
3. Build a `scikit-learn` preprocessing **Pipeline** — fitting transformers **on the training set only** and applying to all splits.
4. Export the feature-engineered datasets and the fitted pipeline for Notebook 6 (Modeling).

> **Key Leakage Prevention Rule**: All `fit()` calls happen **only on `train`**. `val` and `test` are only `transform()`-ed.

In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
from dotenv import load_dotenv
from IPython.display import display

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)
print('Libraries imported successfully.')

Libraries imported successfully.


In [2]:
def find_project_root(start_path: Path) -> Path:
    for parent in [start_path] + list(start_path.parents):
        if (parent / '.env').exists() or (parent / '.git').exists():
            return parent
    return start_path

root_dir = find_project_root(Path.cwd())
load_dotenv(dotenv_path=root_dir / '.env')

split_dir = root_dir / 'tasks' / 'task-02-notebooks' / 'artifacts' / '03_split'
output_dir = root_dir / 'tasks' / 'task-02-notebooks' / 'artifacts' / '05_feature_engineering'
output_dir.mkdir(parents=True, exist_ok=True)

train_raw = pd.read_parquet(split_dir / 'train.parquet')
val_raw   = pd.read_parquet(split_dir / 'val.parquet')
test_raw  = pd.read_parquet(split_dir / 'test.parquet')

print(f'Train: {train_raw.shape}, Val: {val_raw.shape}, Test: {test_raw.shape}')

Train: (67529, 37), Val: (14470, 37), Test: (14471, 37)


## Step 1: Feature Engineering Function

We apply the same deterministic feature derivations to all splits (train, validation, test):
- **Lag features**: `approval_lag_hrs`, `carrier_lag_days`.
- **Temporal features**: `order_hour`, `order_dayofweek`, `order_month`.
- **Geo feature**: `haversine_distance_km` (customer to seller).

> **Operational Prediction-Time Boundary Note**:
> `carrier_lag_days` (`order_delivered_carrier_date - order_purchase_timestamp`) measures dispatch latency. This formulation targets an operational checkpoint at **carrier handover time** to alert logistics teams of orders at severe risk before the estimated delivery SLA lapses. Post-delivery columns (`order_delivered_customer_date`, `actual_delivery_days`, `delivery_delay_days`) are strictly eliminated to prevent forward-looking target leakage.

In [3]:
# Columns to drop before modeling (IDs, leaky post-delivery timestamps, redundant raw columns)
DROP_COLS = [
    'order_id', 'customer_id', 'customer_unique_id', 'primary_seller_id',
    'order_status',
    'order_delivered_customer_date',
    'actual_delivery_days',
    'delivery_delay_days',
    'order_delivered_carrier_date',
    'customer_city', 'primary_seller_city',
    'customer_lat', 'customer_lng', 'seller_lat', 'seller_lng',  # replaced by haversine
    'customer_zip_code_prefix', 'primary_seller_zip_code',        # replaced by state-level features
    'order_purchase_timestamp', 'order_approved_at',              # replaced by derived lag features
    'order_estimated_delivery_date',                              # replaced by estimated_delivery_days
]

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Apply deterministic feature engineering to a raw split."""
    df = df.copy()

    # Temporal features from purchase timestamp
    df['order_hour']       = df['order_purchase_timestamp'].dt.hour
    df['order_dayofweek']  = df['order_purchase_timestamp'].dt.dayofweek
    df['order_month']      = df['order_purchase_timestamp'].dt.month

    # Approval lag: hours from purchase to payment approval
    df['approval_lag_hrs'] = (
        df['order_approved_at'] - df['order_purchase_timestamp']
    ).dt.total_seconds() / 3600.0

    # Carrier lag: days from purchase to carrier dispatch
    df['carrier_lag_days'] = (
        df['order_delivered_carrier_date'] - df['order_purchase_timestamp']
    ).dt.total_seconds() / 86400.0

    # Haversine distance (customer -> seller)
    geo_mask = df[['customer_lat', 'customer_lng', 'seller_lat', 'seller_lng']].notna().all(axis=1)
    df['haversine_distance_km'] = np.nan
    df.loc[geo_mask, 'haversine_distance_km'] = haversine_km(
        df.loc[geo_mask, 'customer_lat'], df.loc[geo_mask, 'customer_lng'],
        df.loc[geo_mask, 'seller_lat'],   df.loc[geo_mask, 'seller_lng']
    )

    # Drop leaky and raw columns
    df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])
    return df

train_fe = engineer_features(train_raw)
val_fe   = engineer_features(val_raw)
test_fe  = engineer_features(test_raw)

# Separate targets
y_train = train_fe.pop('is_late')
y_val   = val_fe.pop('is_late')
y_test  = test_fe.pop('is_late')

print(f'After feature engineering — Train: {train_fe.shape}, Val: {val_fe.shape}, Test: {test_fe.shape}')
print(f'Target — Train late rate: {y_train.mean()*100:.2f}%, Val: {y_val.mean()*100:.2f}%, Test: {y_test.mean()*100:.2f}%')

After feature engineering — Train: (67529, 22), Val: (14470, 22), Test: (14471, 22)
Target — Train late rate: 8.11%, Val: 8.11%, Test: 8.11%


## Step 2: Define Feature Columns & Build Preprocessing Pipeline

We use a `ColumnTransformer` with three branches:
1. **Numeric**: `SimpleImputer(median)` → `StandardScaler`.
2. **High-cardinality categoricals** (`customer_state`, `primary_seller_state`, `primary_product_category`): `SimpleImputer(most_frequent)` → `OneHotEncoder(handle_unknown='ignore')`.
3. **Low-cardinality categorical** (`dominant_payment_type`): `SimpleImputer(most_frequent)` → `OneHotEncoder(handle_unknown='ignore')`.

In [4]:
NUMERIC_COLS = [
    'item_count', 'total_price', 'avg_item_price', 'total_freight',
    'avg_item_freight', 'total_weight_g', 'total_volume_cm3', 'num_sellers',
    'total_payment_value', 'payment_installments_max', 'payment_transactions_count',
    'estimated_delivery_days',
    'order_hour', 'order_dayofweek', 'order_month',
    'approval_lag_hrs', 'carrier_lag_days', 'haversine_distance_km'
]

CATEGORICAL_COLS = [
    'customer_state', 'primary_seller_state',
    'primary_product_category', 'dominant_payment_type'
]

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse=False))
])

preprocessor = ColumnTransformer([
    ('numeric',      numeric_pipeline,      NUMERIC_COLS),
    ('categorical',  categorical_pipeline,  CATEGORICAL_COLS)
], remainder='drop')

print('Preprocessing pipeline defined.')
print(f'Numeric features    : {len(NUMERIC_COLS)}')
print(f'Categorical features: {len(CATEGORICAL_COLS)}')

Preprocessing pipeline defined.
Numeric features    : 18
Categorical features: 4


## Step 3: Fit on Train, Transform All Splits

In [5]:
# Fit on train only
preprocessor.fit(train_fe)

# Transform all splits
X_train = preprocessor.transform(train_fe)
X_val   = preprocessor.transform(val_fe)
X_test  = preprocessor.transform(test_fe)

# Recover feature names
ohe_cat_features = preprocessor.named_transformers_['categorical']['encoder'].get_feature_names_out(CATEGORICAL_COLS)
all_feature_names = NUMERIC_COLS + list(ohe_cat_features)

X_train_df = pd.DataFrame(X_train, columns=all_feature_names)
X_val_df   = pd.DataFrame(X_val,   columns=all_feature_names)
X_test_df  = pd.DataFrame(X_test,  columns=all_feature_names)

print(f'X_train shape: {X_train_df.shape}')
print(f'X_val shape  : {X_val_df.shape}')
print(f'X_test shape : {X_test_df.shape}')
print(f'Total features: {len(all_feature_names)}')
display(X_train_df.head(3))

X_train shape: (67529, 145)
X_val shape  : (14470, 145)
X_test shape : (14471, 145)
Total features: 145


,item_count,total_price,avg_item_price,total_freight,avg_item_freight,total_weight_g,total_volume_cm3,num_sellers,total_payment_value,payment_installments_max,payment_transactions_count,estimated_delivery_days,order_hour,order_dayofweek,order_month,approval_lag_hrs,carrier_lag_days,haversine_distance_km,customer_state_AC,customer_state_AL,customer_state_AM,customer_state_AP,customer_state_BA,customer_state_CE,customer_state_DF,...,primary_product_category_luggage_accessories,primary_product_category_market_place,primary_product_category_music,primary_product_category_musical_instruments,primary_product_category_office_furniture,primary_product_category_party_supplies,primary_product_category_pc_gamer,primary_product_category_perfumery,primary_product_category_pet_shop,primary_product_category_portateis_cozinha_e_preparadores_de_alimentos,primary_product_category_security_and_services,primary_product_category_signaling_and_security,primary_product_category_small_appliances,primary_product_category_small_appliances_home_oven_and_coffee,primary_product_category_sports_leisure,primary_product_category_stationery,primary_product_category_tablets_printing_image,primary_product_category_telephony,primary_product_category_toys,primary_product_category_unknown,primary_product_category_watches_gifts,dominant_payment_type_boleto,dominant_payment_type_credit_card,dominant_payment_type_debit_card,dominant_payment_type_voucher
0,-0.264762,-0.423238,-0.406254,-0.345872,-0.319178,-0.389584,-0.470132,-0.112767,-0.439114,0.026725,-0.124925,-0.180465,0.790341,-0.384414,0.609403,-0.482548,-0.687330,-0.416919,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,-0.264762,-0.470641,-0.458566,-0.270339,-0.213374,-0.447734,-0.495240,-0.112767,-0.476775,-0.711242,-0.124925,1.064344,0.977638,-0.384414,1.847228,-0.180918,-0.659231,0.219567,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,7.279174,-0.551226,-0.637472,0.636055,-0.805370,-0.447734,-0.255448,-0.112767,-0.462471,-0.711242,-0.124925,-1.950480,-1.082626,-0.893006,0.609403,-0.488599,-0.866404,-0.939266,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


## Step 4: Save Artifacts

In [6]:
# Feature datasets
X_train_df.to_parquet(output_dir / 'X_train.parquet', index=False)
X_val_df.to_parquet(output_dir / 'X_val.parquet', index=False)
X_test_df.to_parquet(output_dir / 'X_test.parquet', index=False)

# Target series
y_train.reset_index(drop=True).to_frame().to_parquet(output_dir / 'y_train.parquet', index=False)
y_val.reset_index(drop=True).to_frame().to_parquet(output_dir / 'y_val.parquet', index=False)
y_test.reset_index(drop=True).to_frame().to_parquet(output_dir / 'y_test.parquet', index=False)

# Fitted pipeline
pipeline_path = output_dir / 'preprocessing_pipeline.joblib'
joblib.dump(preprocessor, pipeline_path)

# Feature names list
feature_list_path = output_dir / 'feature_names.txt'
with open(feature_list_path, 'w') as f:
    f.write('\n'.join(all_feature_names))

print(f'✓ X_train.parquet, X_val.parquet, X_test.parquet saved.')
print(f'✓ y_train.parquet, y_val.parquet, y_test.parquet saved.')
print(f'✓ preprocessing_pipeline.joblib saved: {pipeline_path.stat().st_size / 1024:.1f} KB')
print(f'✓ feature_names.txt saved ({len(all_feature_names)} features).')
print('\n=======================================================')
print('✓ Notebook 5 (Feature Engineering) completed successfully!')
print('=======================================================')

✓ X_train.parquet, X_val.parquet, X_test.parquet saved.
✓ y_train.parquet, y_val.parquet, y_test.parquet saved.
✓ preprocessing_pipeline.joblib saved: 7.9 KB
✓ feature_names.txt saved (145 features).

✓ Notebook 5 (Feature Engineering) completed successfully!
